In [ ]:
import numpy as np
import torch

from protossl.datasets import HeedbECGDataset

dataset_path = "/opt/gpudata/ecg/heedb"

In [3]:
ds_train = HeedbECGDataset(
    dataset_path=dataset_path,
    split="train",
    sampling_rate=100,
    heedb_split_type="by-label",
)
ds_val = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    heedb_split_type="by-label",
)
ds_test = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    heedb_split_type="by-label",
)

================get_heedb_metadata=================
reading HEEDB metadata
read MGB data
read Emory data
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 8163230/8163230 [01:17<00:00, 105109.73it/s]


Of 8163230 ECGs and 11607259 annotations, 8163229 matched (1 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 453512/453512 [00:02<00:00, 225403.89it/s]


Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 453512/453512 [00:04<00:00, 108876.87it/s]

Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly


In [ ]:
def find_tol(pairs: list[tuple[torch.Tensor, torch.Tensor]]) -> float:
    last_tol = np.nan
    tols = np.outer(10.0 ** -np.arange(1, 5), (1, 0.5, 0.25)).ravel() # [0.1, 0.05, 0.025, 0.01, ...]
    for tol in tols:
        try:
            for a, b in pairs:
                assert np.isclose(a, b, atol=tol).all()
        except AssertionError:
            return last_tol
        last_tol = tol
    return last_tol

In [42]:
train_prev = ds_train.labels.sum(axis=0) / len(ds_train)
val_prev = ds_val.labels.sum(axis=0) / len(ds_val)
test_prev = ds_test.labels.sum(axis=0) / len(ds_test)

tol = find_tol([
    (train_prev, val_prev),
    (train_prev, test_prev),
    (val_prev, test_prev),
])
print(f"Label prevalences are preserved to {tol*100:0.1f}% tolerance")

Label prevalences are preserved to 0.1% tolerance


In [41]:
train_cooc = ds_train.get_label_cooccurrence()
val_cooc = ds_val.get_label_cooccurrence()
test_cooc = ds_test.get_label_cooccurrence()

tol = find_tol([
    (train_cooc, val_cooc),
    (train_cooc, test_cooc),
    (val_cooc, test_cooc),
])
print(f"Cooccurrence frequencies are preserved to {tol*100:0.1f}% tolerance")

Cooccurrence frequencies are preserved to 1.0% tolerance
